In [50]:
# On-Demand 
# 데이터의 개수가 런타임에서 결정 
# 병렬처리가 필요할 때 여러문서를 동시에 요약 
# Map-Reduce 패턴 각 workers 결과를 모아 최종 처리 


# from langgraph.types import Send

# class State(TypedDict):
#     items: list[str]
#     results: Annotated[list, operator.add]  # 병렬 결과 수집

# def orchestrator(state: State):
#     # 데이터 개수에 따라 on-demand로 worker 생성
#     return [Send("worker", {"item": item}) for item in state["items"]]

# def worker(state: dict):
#     item = state["item"]
#     print(f"처리 중: {item}")
#     return {"results": [f"{item}_완료"]}

# graph_builder.add_node("orchestrator", orchestrator)
# graph_builder.add_node("worker", worker)

# graph_builder.add_conditional_edges("orchestrator", lambda s: s, ["worker"])
# graph_builder.add_edge(START, "orchestrator")
# graph_builder.add_edge("worker", END)

# graph = graph_builder.compile()

# graph.invoke({"items": ["사과", "바나나", "딸기"]})
# worker 3개가 병렬로 동시 실행됨


In [51]:
from langgraph.graph import StateGraph, START, END
from typing import Union
from typing_extensions import TypedDict , Annotated
from langgraph.types import Send
import operator



In [52]:
class State(TypedDict):
    words: list[str]
    output :Annotated[list[dict[str, Union[str,int]]],operator.add]

graph_builder = StateGraph(State)


In [53]:
def node_one(state:State):
    print(f"I want to count {len(state['words'])} words in my state")
    # return {}

# def node_two(state:State):
#     print("node_two->", state)
#     return {}

def node_two(word: str):
    return {
        "output": [
            {
                "word": word,
                "letters": len(word)
            }
        ]
    }





In [54]:
graph_builder.add_node("node_one",node_one)
graph_builder.add_node("node_two",node_two)


def dispatcher(state:State):
    # words = state["words"]
    # Send=[]
    # for word in words:
    #     Send.append(Send("node_two",word))
    # return Send

    return [Send("node_two", word) for word in state["words"]]
    

graph_builder.add_edge(START,"node_one")
# graph_builder.add_edge(f"node_one","node_two")
graph_builder.add_conditional_edges("node_one",dispatcher,["node_two"])
graph_builder.add_edge("node_two",END)

graph = graph_builder.compile()
graph.invoke({
    "words" : ["hello","world","how","are","you","doing"],
})



I want to count 6 words in my state


{'words': ['hello', 'world', 'how', 'are', 'you', 'doing'],
 'output': [{'word': 'hello', 'letters': 5},
  {'word': 'world', 'letters': 5},
  {'word': 'how', 'letters': 3},
  {'word': 'are', 'letters': 3},
  {'word': 'you', 'letters': 3},
  {'word': 'doing', 'letters': 5}]}